This pipeline test uses:
- **Count and area:** OpenCV
- **Stage Classificator:** EfficientNet-B0
- **Action Motor:** LLaMA-3-8B (Text Only)

In [1]:
from pathlib import Path
import sys

In [2]:
project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "utils" / "utls_install_libs.py").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not find utils/utls_install_libs.py")

sys.path.insert(0, str(project_root))

In [3]:
from utils.utls_install_libs import install_missing_libs
from utils.utls_work_images import build_dataset

In [4]:
import time
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

In [5]:
print("📥 [1/4] Baixando imagens do Google Drive...")
X_exp, y_exp = build_dataset(limit_per_class_collected=50, folder_name_collected="expulsion")
X_inc, y_inc = build_dataset(limit_per_class_collected=50, folder_name_collected="incubation")

X_completo = np.concatenate((X_exp, X_inc), axis=0)
y_completo = np.concatenate((y_exp, y_inc), axis=0)
X_train, X_test, y_train, y_test = train_test_split(X_completo, y_completo, test_size=0.2, random_state=42)

📥 [1/4] Baixando imagens do Google Drive...


In [6]:
print("🧠 [2/4] Baixando pesos do EfficientNet-B0 e aplicando Transfer Learning...")
# Trocamos o ResNet pelo EfficientNet (mais rápido e otimizado)
efficient_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
efficient_base.trainable = False 

modelo_efficient = Sequential([
    efficient_base,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') 
])

modelo_efficient.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Treinamos 5 épocas igual ao ResNet para ser uma comparação justa
modelo_efficient.fit(X_train, y_train, epochs=5, verbose=0)

🧠 [2/4] Baixando pesos do EfficientNet-B0 e aplicando Transfer Learning...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [7]:
def calcular_area_bolhas_cv2(img_array):
    img_uint8 = (img_array * 255).astype(np.uint8)
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    bordas = cv2.Canny(gray, 50, 150)
    contornos, _ = cv2.findContours(bordas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    qtd_bolhas = len(contornos)
    percentual_vapor = (sum(cv2.contourArea(c) for c in contornos) / (224 * 224)) * 100
    return qtd_bolhas, round(percentual_vapor, 2)

In [8]:
def simulador_llama3_texto(classe_ia, qtd, pct):
    estado = "Expulsão crítica" if classe_ia == 0 else "Incubação inicial"
    
    # Este é o prompt exato que vai para o LLM
    prompt_injetado = (f"O modelo visual detectou {estado}. "
                       f"A matemática acusa {qtd} bolhas e {pct}% de vapor.")
    
    # Resposta simulada baseada na capacidade de raciocínio de um LLM puro de texto
    if classe_ia == 0 and pct < 20:
        return "LLaMA: Discrepância notada. Visão indica expulsão, mas % de vapor é baixa. Ignorar OpenCV. Acionar resfriamento moderado."
    elif classe_ia == 0 and pct >= 20:
        return "LLaMA: Expulsão validada matematicamente. Acionamento de emergência total."
    elif classe_ia == 1 and qtd > 300:
        return "LLaMA: Alerta de turbulência. Muitas bolhas de incubação detectadas. Reduzir 10% da potência."
    else:
        return "LLaMA: Sistema estável. Monitoramento em modo passivo."

In [9]:
resultados_p3 = []
print("⏱️ [3/4] Iniciando cronometragem do Pipeline 03 (EfficientNet-B0)...")

for i in range(len(X_test)):
    img = X_test[i]
    inicio_timer = time.perf_counter()
    
    # 1. OpenCV
    qtd_bolhas, pct_vapor = calcular_area_bolhas_cv2(img)
    
    # 2. EfficientNet-B0
    img_batch = np.expand_dims(img, axis=0) 
    probabilidade = modelo_efficient.predict(img_batch, verbose=0)[0][0]
    classe_ia = 1 if probabilidade > 0.5 else 0 
    
    # 3. LLM Texto
    acao_sugerida = simulador_llama3_texto(classe_ia, qtd_bolhas, pct_vapor)
    
    fim_timer = time.perf_counter()
    latencia_ms = round((fim_timer - inicio_timer) * 1000, 2)
    
    resultados_p3.append({
        "ID_Teste": i,
        "Real": "Incubação" if y_test[i] == 1 else "Expulsão",
        "Predicao_EfficientNet": "Incubação" if classe_ia == 1 else "Expulsão",
        "Acao_Sugerida": acao_sugerida,
        "Latencia_ms": latencia_ms
    })

⏱️ [3/4] Iniciando cronometragem do Pipeline 03 (EfficientNet-B0)...


In [10]:
df_p3 = pd.DataFrame(resultados_p3)
df_p3.to_csv('benchmark_pipeline_03.csv', index=False)

print(f"\n✅ [4/4] Benchmark concluído! Salvo como 'benchmark_pipeline_03.csv'.")
print(f"📊 Latência Média do Pipeline 03: {df_p3['Latencia_ms'].mean():.2f} ms")
display(df_p3.head())


✅ [4/4] Benchmark concluído! Salvo como 'benchmark_pipeline_03.csv'.
📊 Latência Média do Pipeline 03: 269.99 ms


,ID_Teste,Real,Predicao_EfficientNet,Acao_Sugerida,Latencia_ms
0,0,Incubação,Incubação,LLaMA: Sistema estável. Monitoramento em modo ...,2598.62
1,1,Incubação,Incubação,LLaMA: Sistema estável. Monitoramento em modo ...,148.64
2,2,Incubação,Incubação,LLaMA: Sistema estável. Monitoramento em modo ...,154.11
3,3,Expulsão,Incubação,LLaMA: Sistema estável. Monitoramento em modo ...,146.41
4,4,Expulsão,Incubação,LLaMA: Alerta de turbulência. Muitas bolhas de...,148.29
